In [3]:
from transformers import TimesformerModel, TimesformerConfig

config = TimesformerConfig.from_pretrained("facebook/timesformer-base-finetuned-k400")
model = TimesformerModel.from_pretrained("facebook/timesformer-base-finetuned-k400", config=config)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

In [4]:
print(config)


TimesformerConfig {
  "architectures": [
    "TimesformerForVideoClassification"
  ],
  "attention_probs_dropout_prob": 0.0,
  "attention_type": "divided_space_time",
  "drop_path_rate": 0,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "id2label": {
    "0": "abseiling",
    "1": "air drumming",
    "2": "answering questions",
    "3": "applauding",
    "4": "applying cream",
    "5": "archery",
    "6": "arm wrestling",
    "7": "arranging flowers",
    "8": "assembling computer",
    "9": "auctioning",
    "10": "baby waking up",
    "11": "baking cookies",
    "12": "balloon blowing",
    "13": "bandaging",
    "14": "barbequing",
    "15": "bartending",
    "16": "beatboxing",
    "17": "bee keeping",
    "18": "belly dancing",
    "19": "bench pressing",
    "20": "bending back",
    "21": "bending metal",
    "22": "biking through snow",
    "23": "blasting sand",
    "24": "blowing glass",
    "25": "blowing leaves",
    "26": "blowing nose",
    

In [5]:
import os
import cv2
import torch
import numpy as np
from torchvision import transforms
from transformers import TimesformerModel, TimesformerConfig

In [6]:
# Paths
video_folder = "/content/drive/MyDrive/video_captioning"
output_feature_folder = "/content/drive/MyDrive/video_captioning/msvd_features"

os.makedirs(output_feature_folder, exist_ok=True)

In [7]:
# Load pretrained TimeSformer model & config (frozen)
model_name = "facebook/timesformer-base-finetuned-k400"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading pretrained TimeSformer...")
config = TimesformerConfig.from_pretrained(model_name)
model = TimesformerModel.from_pretrained(model_name, config=config)
model.eval()
model.to(device)

Loading pretrained TimeSformer...


TimesformerModel(
  (embeddings): TimesformerEmbeddings(
    (patch_embeddings): TimesformerPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (time_drop): Dropout(p=0.0, inplace=False)
  )
  (encoder): TimesformerEncoder(
    (layer): ModuleList(
      (0-11): 12 x TimesformerLayer(
        (drop_path): Identity()
        (attention): TimeSformerAttention(
          (attention): TimesformerSelfAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
          )
          (output): TimesformerSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): TimesformerIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (dropout): Dropout(p=0.0, inpla

In [8]:
# ImageNet normalization stats
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

In [9]:
def sample_frames(video_path, num_frames=8):
    """Sample num_frames uniformly from the video."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < num_frames:
        # If fewer frames, duplicate last frame to pad
        frame_idxs = list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)
    else:
        frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if i in frame_idxs:
            frames.append(frame)
    cap.release()
    return frames

In [13]:
def preprocess_frames(frames):
    """Apply preprocessing to frames and stack into tensor."""
    processed = [preprocess(frame) for frame in frames]  # list of [3,224,224]
    video_tensor = torch.stack(processed, dim=0)  # shape [T,3, H, W]
    return video_tensor.unsqueeze(0)  # [1,T,3, H, W]


Testing for single video

In [11]:
# def extract_features(video_id):
#     video_path = os.path.join(video_folder, f"{video_id}.mp4")  # adjust extension if needed
#     if not os.path.exists(video_path):
#         print(f"Video not found: {video_path}")
#         return None

#     frames = sample_frames(video_path, num_frames=config.num_frames)
#     input_tensor = preprocess_frames(frames).to(device)

#     with torch.no_grad():
#         outputs = model(input_tensor)

#     # outputs.last_hidden_state shape: [batch_size, sequence_length, hidden_size]
#     # We can take the [CLS] token embedding at position 0
#     cls_embedding = outputs.last_hidden_state[:, 0, :]  # take  cls token [1, hidden_size]

#     return cls_embedding.squeeze(0).cpu()  # [hidden_size]





# # Example usage
# video_ids = ["video0"]

# for vid in video_ids:
#     features = extract_features(vid)
#     if features is not None:
#         save_path = os.path.join(output_feature_folder, f"{vid}.pt")
#         torch.save(features, save_path)
#         print(f"Saved features for {vid} to {save_path}")


For full video folder

In [14]:
def extract_and_save_features(video_id):
    video_path = os.path.join(video_folder, f"{video_id}.mp4")  # Adjust extension if needed
    if not os.path.exists(video_path):
        print(f"Video not found: {video_path}")
        return

    feature_path = os.path.join(output_feature_folder, f"{video_id}.pt")
    if os.path.exists(feature_path):
        print(f"Features already exist for {video_id}, skipping.")
        return

    frames = sample_frames(video_path, num_frames=config.num_frames)
    input_tensor = preprocess_frames(frames).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token embedding

    torch.save(cls_embedding.squeeze(0).cpu(), feature_path)
    print(f"Saved features for {video_id} to {feature_path}")

# List all video IDs by scanning your video folder
video_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
video_ids = [os.path.splitext(f)[0] for f in video_files]

# Loop over all videos and extract features
for vid in video_ids:
    extract_and_save_features(vid)

Saved features for video0 to /content/drive/MyDrive/video_captioning/msvd_features/video0.pt


In [15]:
import torch

features = torch.load('/content/drive/MyDrive/video_captioning/msvd_features/video0.pt')
print(features.shape)
print(features)


torch.Size([768])
tensor([ 8.0466e-02, -1.2600e+00,  9.7402e-01, -4.9036e-01,  1.2072e-02,
         1.3738e+00, -1.6375e+00,  7.0694e-01, -6.1103e-02, -7.6208e-01,
        -9.2872e-01, -3.2817e+00,  1.8331e+00,  8.2797e-01,  6.0379e-02,
         4.3965e-02, -8.6345e-01,  8.8358e-01,  4.8553e-01,  1.9373e+00,
        -5.7097e-01, -1.0269e+00, -1.3801e-01, -5.1339e-02, -3.8743e-01,
         3.5101e-02, -2.7045e-02, -5.1492e-02,  2.3589e-01, -1.2492e+00,
        -6.7989e-01,  6.1940e-01,  2.2338e-01, -9.1453e-01, -4.7102e-01,
        -5.3699e-01,  6.6211e-01, -1.5484e+00,  3.5440e-01, -8.5574e-01,
         1.0406e-01, -9.5731e-01,  7.1434e-01,  1.7041e+00,  2.4268e-01,
        -4.1314e-01, -1.5460e+00, -1.5583e+00, -5.1038e-01, -4.3007e-01,
         7.6903e-01, -2.8389e-01, -4.8304e-02, -1.8757e+00, -6.1528e-01,
        -1.0035e+00,  2.4125e-01,  1.4616e+00, -5.9648e-01, -4.1297e-01,
         1.1425e+00, -6.6811e-01,  4.9970e-01, -4.0883e-02,  4.4602e-01,
        -5.6595e-01,  4.5735e-01,

Testing the video classification to see if the model is working properly or not  

In [17]:
import os
import cv2
import torch
import numpy as np
from torchvision import transforms
from transformers import TimesformerForVideoClassification, TimesformerConfig
import requests

# --- Paths ---
video_folder = "/content/drive/MyDrive/video_captioning"
video_filename = "video19.mp4"  # change to your video file
video_path = os.path.join(video_folder, video_filename)

# --- Load Kinetics-400 Labels ---
# Download kinetics labels file (if not present)
labels_url = "https://raw.githubusercontent.com/deepmind/kinetics-i3d/master/data/label_map.txt"
labels_path = "kinetics_labels.txt"
if not os.path.exists(labels_path):
    r = requests.get(labels_url)
    with open(labels_path, "w") as f:
        f.write(r.text)
with open(labels_path, "r") as f:
    kinetics_labels = [line.strip() for line in f.readlines()]

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model ---
model_name = "facebook/timesformer-base-finetuned-k400"
print("Loading Timesformer classification model...")
model = TimesformerForVideoClassification.from_pretrained(model_name)
model.eval()
model.to(device)

# --- Preprocessing ---
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

def sample_frames(video_path, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < num_frames:
        frame_idxs = list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)
    else:
        frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if i in frame_idxs:
            frames.append(frame)
    cap.release()
    return frames

def preprocess_frames(frames):
    processed = [preprocess(frame) for frame in frames]  # list of [3,224,224]
    video_tensor = torch.stack(processed, dim=0)  # [T,3,H,W]
    return video_tensor.unsqueeze(0)  # [1,T,3,H,W]

# --- Run Classification ---
frames = sample_frames(video_path, num_frames=8)
input_tensor = preprocess_frames(frames).to(device)

with torch.no_grad():
    outputs = model(input_tensor)
    logits = outputs.logits  # [1, 400]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    top5_prob, top5_catid = torch.topk(probs, 5)

print("Top 5 Kinetics-400 predictions:")
for i in range(top5_prob.size(1)):
    print(f"{kinetics_labels[top5_catid[0,i]]}: {top5_prob[0,i].item()*100:.2f}%")


Loading Timesformer classification model...
Top 5 Kinetics-400 predictions:
playing tennis: 76.85%
playing cricket: 4.13%
playing badminton: 2.16%
throwing ball: 1.48%
sweeping floor: 1.38%
